<a href="https://colab.research.google.com/github/ahmedkhanpakistani5-hash/-intern-feedback-to-understand-positive-and-negative-sentiments./blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedkhanpakistani5-hash/ahmad-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import pandas as pd
from huggingface_hub import hf_hub_download

MONTH = "2026-03"
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename=f"fact_content_daily_performance/month={MONTH}/data_0.parquet",
    token=HF_TOKEN,
)
df = pd.read_parquet(path)
print("rows:", len(df))

rows: 9841378


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row = one content item (content_hash_id) for one client (client_hash_id) on one day (report_date), over March 2026 (month=2026-03).

In [ ]:
GRAIN = ["report_date", "client_hash_id", "content_hash_id"]
print("rows:", len(df))
print("date span:", df["report_date"].min(), "to", df["report_date"].max())
print("distinct days:", df["report_date"].nunique())
print("distinct clients:", df["client_hash_id"].nunique())
print("distinct content items:", df["content_hash_id"].nunique())
print("duplicate rows on grain:", df.duplicated(GRAIN).sum())


rows: 9841378
date span: 2026-03-01 to 2026-03-31
distinct days: 31
distinct clients: 55
distinct content items: 331437
duplicate rows on grain: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Feature(s): [metric columns from an EARLIER window, e.g. last-7-day values] (knowable at the decision moment because they come from days before the prediction)
- Label: [outcome column, e.g. clicks] over the future window (the outcome window)
- Context: report_date, client_hash_id, content_hash_id (used to group and join, not as model inputs)
- Excluded:
  - [same-day outcome column]: Why? Measured in the same window as the label, so it would leak the answer.
  - [column with high % null]: Why? [its % null from the table above], so it isn't reliably available.

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "pct_null": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(),
})
print(summary)


                            dtype  pct_null  n_unique
report_date                object       0.0        31
client_hash_id             object       0.0        55
content_hash_id            object       0.0    331437
client_has_gsc               bool       0.0         1
client_has_ga4               bool       0.0         2
gsc_data_available           bool       0.0         2
ga4_data_available         object      30.7         2
gsc_impressions             int64       0.0      4854
gsc_clicks                  int64       0.0       140
gsc_sum_position            int64       0.0     34343
gsc_avg_position          float64      63.3    457684
ga4_pageviews             float64      30.7       234
ga4_sessions              float64      30.7       219
ga4_users                 float64      30.7       224
ga4_engaged_sessions      float64      30.7        18
ga4_total_engagement_sec  float64      30.7      1196
sessions_organic          float64      30.7       125
sessions_direct           fl

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


- Grain: how many keys (report_date, client_hash_id, content_hash_id) appear more than once.
- Counts: row count and date span for month=2026-03.
- Availability: how many rows survive when filtering with IS TRUE on client_has_gsc and client_has_ga4.

In [ ]:
!pip install -q duckdb
import duckdb
con = duckdb.connect()
con.execute(f"CREATE VIEW march AS SELECT * FROM read_parquet('{path}')")


In [ ]:
con.sql("""
SELECT COUNT(*) AS keys_with_duplicates FROM (
  SELECT 1 FROM march
  GROUP BY report_date, client_hash_id, content_hash_id
  HAVING COUNT(*) > 1
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┐
│ keys_with_duplicates │
│        int64         │
├──────────────────────┤
│                    0 │
└──────────────────────┘



In [ ]:
con.sql("""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day,
       MAX(report_date) AS last_day, COUNT(DISTINCT report_date) AS n_days
FROM march
""").show()

┌─────────┬────────────┬────────────┬────────┐
│ n_rows  │ first_day  │  last_day  │ n_days │
│  int64  │    date    │    date    │ int64  │
├─────────┼────────────┼────────────┼────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │     31 │
└─────────┴────────────┴────────────┴────────┘



In [ ]:
con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS gsc_rows,
       COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) AS ga4_rows
FROM march
""").show()

┌────────────┬──────────┬──────────┐
│ total_rows │ gsc_rows │ ga4_rows │
│   int64    │  int64   │  int64   │
├────────────┼──────────┼──────────┤
│    9841378 │  9841378 │  6822637 │
└────────────┴──────────┴──────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: this slice cannot tell me [what], because [number from the output, e.g. only X% of rows come from clients with GA4 access].
Effect on modeling: [e.g. a few large clients supply most rows, so the model may learn them instead of the average client].

In [ ]:
print("% null per column (top 10):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

per_client = df["client_hash_id"].value_counts()
print("\nclients:", per_client.size)
print("rows per client: min", per_client.min(), "| median", per_client.median(), "| max", per_client.max())

print("\nshare of rows with client_has_gsc True:", round(df["client_has_gsc"].mean() * 100, 1), "%")
print("share of rows with client_has_ga4 True:", round(df["client_has_ga4"].mean() * 100, 1), "%")


% null per column (top 10):
gsc_avg_position      63.3
ga4_data_available    30.7
ga4_sessions          30.7
ga4_pageviews         30.7
ga4_users             30.7
ai_chatgpt            30.7
sessions_ai           30.7
sessions_paid         30.7
sessions_social       30.7
sessions_referral     30.7
dtype: float64

clients: 55
rows per client: min 341 | median 83119.0 | max 988497

share of rows with client_has_gsc True: 100.0 %
share of rows with client_has_ga4 True: 69.3 %


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.